# Частотное распределение охвата для открытых и закрытых интервалов
Пример выгрузки охватов в зависимости от количества контактов с рекламой

## Описание задачи и условий расчета
- Регион: Россия 0+
- Период: 01.07.2024-07.07.2024
- ЦА: Все 4+, Все 14-59, Ж 25-59
- Место просмотра: Все места (Дом, Дача)
- Каналы: все каналы проекта TV Index
- Ролики: Статус события - Реальный, Рекламодатель - ИНТЕРНЕТ РЕШЕНИЯ, Ролик тип - Ролик
- Статистики: CumReach%, FrequencyDistribution% (1+, 3+, 5+, 1-3, 4-7, 8-20)

## Инициализация

При построении отчета первый шаг в любом ноутбуке - загрузка библиотек, которые помогут обращаться к TVI API и работать с данными.

Выполните следующую ячейку, для этого перейдите в нее и нажмите Ctrl+Enter

In [ ]:
%reload_ext autoreload
%autoreload 2

import sys
import os
import re
import json
import datetime
import time
import pandas as pd
#import matplotlib.pyplot as plt
from IPython.display import JSON
from mediascope_api.core import utils

from mediascope_api.core import net as mscore
from mediascope_api.mediavortex import tasks as cwt
from mediascope_api.mediavortex import catalogs as cwc

# Настраиваем отображение

# Включаем отображение всех колонок
pd.set_option('display.max_columns', None)
# Задаем максимальное количество выводимых строк. Раскомментируйте нужную строку
# 200 строк
# pd.set_option("display.max_rows", 200)
# Отображаем все строки. ВАЖНО! Отображение большого DataFrame требует много ресурсов
# pd.set_option("display.max_rows", None)

# Cоздаем объекты для работы с TVI API
mnet = mscore.MediascopeApiNetwork()
mtask = cwt.MediaVortexTask()
cats = cwc.MediaVortexCats()

## Справочники

Получим идентификаторы, которые будут использоваться для формирования условий расчета

In [ ]:
# Демография - женщины
cats.get_tv_demo_attribute(value_names=['Женщины'])

In [ ]:
# Статус события - реальный
cats.get_tv_issue_status(name=['Реальный'])

In [ ]:
# Рекламодатель - ИНТЕРНЕТ РЕШЕНИЯ
cats.get_tv_advertiser(name=['ИНТЕРНЕТ РЕШЕНИЯ'])

In [ ]:
# Ролик тип - ролик
cats.get_tv_ad_type(name=['ролик'])

## Формирование задания
Зададим условия расчета

In [ ]:
# Задаем период
# Период указывается в виде списка ('Начало', 'Конец') в формате 'YYYY-MM-DD'. Можно указать несколько периодов
date_filter = [('2024-07-01', '2024-07-07')]

# Задаем дни недели
weekday_filter = None

# Задаем тип дня
daytype_filter = None

# Задаем ЦА
basedemo_filter = None

# Доп фильтр ЦА, нужен только в случае расчета отношения между ЦА, например, при расчете Affinity Index
targetdemo_filter = None

# Задаем место просмотра
location_filter=None

# Задаем каналы
company_filter = None

# Указываем фильтр программ
program_filter = None

# Фильтр блоков:
break_filter = None

# Фильтр роликов: Статус события - Реальный, Рекламодатель - ИНТЕРНЕТ РЕШЕНИЯ, Ролик тип - Ролик
ad_filter = 'adIssueStatusId = R and advertiserId = 515109 and and adTypeId = 1'

# Указываем список срезов (группировок)
slices = [
    'advertiserName', # Рекламодатель
    'tvNetName', # Национальная телекомпания
    'researchDate' # Дата (дни)
]

# Указываем список статистик для расчета
statistics = [
    'CumReachPer',
    'FrequencyDistPer'
]

# Задаем условия расчета Frequency Distribution
frequency_dist_conditions = {
    "intervals": [
        {'from': 1}, 
        {'from': 3}, 
        {'from': 5}, 
        {'from': 1, 'to': 3}, 
        {'from': 4, 'to': 7}, 
        {'from': 8, 'to': 20},
    ]
}

# Задаем условия сортировки
sortings = {'tvNetName':'ASC', 'researchDate':'ASC'}

# Задаем опции расчета
options = {
    "issueType": "AD", #Тип события - Ролики
    "kitId": 1, #TV Index Russia all
    "baseDate": None, #Базовый день
    "baseDateCalcType": "BY_RESEARCH_PERIOD", #Автоматический базовый день - по периоду
    "oneBaseDatePerEachDateRange": False, #для всех периодов-срезов базовый день - средний день периода, заданного в календаре
    "useNbd": True #НБД коррекция
}

Сформируем словарь с целевыми аудиториями

In [ ]:
# Задаем необходимые группы
targets = {
    'a. Все 4+': 'age >= 4',
    'b. Все 14-59': 'age >= 14 and age <= 59',
    'c. Ж 25-59': 'sex = 2 and age >= 25 and age <= 59',
}

## Расчет задания

In [ ]:
# Посчитаем задания в цикле
tasks = []
print("Отправляем задания на расчет")

# Для каждой ЦА формируем задание и отправляем на расчет
for target, syntax in targets.items():
    
    # Подставляем значения словаря в параметры
    project_name = target 
    basedemo_filter = syntax
      
    # Формируем задание для API TV Index в формате JSON
    task_json = mtask.build_crosstab_task(date_filter=date_filter, weekday_filter=weekday_filter, 
                                          daytype_filter=daytype_filter, company_filter=company_filter, 
                                          location_filter=location_filter, basedemo_filter=basedemo_filter, 
                                          targetdemo_filter=targetdemo_filter,program_filter=program_filter, 
                                          break_filter=break_filter, ad_filter=ad_filter, slices=slices, 
                                          statistics=statistics, frequency_dist_conditions=frequency_dist_conditions,
                                          sortings=sortings, options=options)

    # Для каждого этапа цикла формируем словарь с параметрами и отправленным заданием на расчет
    tsk = {}
    tsk['project_name'] = project_name    
    tsk['task'] = mtask.send_crosstab_task(task_json)
    tasks.append(tsk)
    time.sleep(2)
    print('.', end = '')
    
print(f"\nid: {[i['task']['taskId'] for i in tasks]}") 

print('')
# Ждем выполнения
print('Ждем выполнения')
tsks = mtask.wait_task(tasks)
print('Расчет завершен, получаем результат')

# Получаем результат
results = []
print('Собираем таблицу')
for t in tasks:
    tsk = t['task'] 
    df_result = mtask.result2table(mtask.get_result(tsk), project_name = t['project_name'])        
    results.append(df_result)
    print('.', end = '')

df = pd.concat(results)

# Приводим порядок столбцов в соответствие с условиями расчета
df = df[['prj_name']+slices+['frequencyDistInterval']+statistics]

df

## Настройка внешнего вида таблицы
Пропустите этот шаг, если хотите экспортировать таблицу в ее текущем виде

In [ ]:
# Формируем сводную таблицу: строки - срезы, столбцы - ЦА, значения - статистики
df = pd.pivot_table(df,
                    values=['CumReachPer','FrequencyDistPer'],
                    index=slices, 
                    columns=['prj_name','frequencyDistInterval'])
df

In [ ]:
# меняем порядок уровней в столбцах
df = df.reorder_levels([1, 0, 2], axis=1).sort_index(axis=1)
df

## Сохраняем в Excel
По умолчанию файл сохраняется в папку `excel`

In [ ]:
df_info = mtask.task_builder.get_report_info()

with pd.ExcelWriter(mtask.task_builder.get_excel_filename('crosstab_spots_03_frequency_distribution_n+')) as writer:
    df.to_excel(writer, 'Report', index=True)
    df_info.to_excel(writer, 'Info', index=False)